# 5G Network Intelligence — Multi-Corpus Spatial Encoder Pretraining

## Hypothesis
Can longer representation pretraining on audited radio and network traffic prefixes improve original DLPRB forecasting beyond an equal-compute target-self-pretraining control? Transfer only the temporal encoder; learn the cell interaction and physical DLPRB output on the original fit prefix.


## Frozen temporal protocol
Fit/stop/score cutoffs: 335/383/443 and 395/443/503. Context 168 hours, four KPIs and missing masks, horizons 1/6/24h. All 2,880 beams, grouped by known station/cell IDs into 90 cells of 32 beams. No future actual inputs, fitted per-beam statistics or identity embeddings. Original values from hour 504 onward are redacted in cloud inputs.


In [ ]:
# Safe review: Run All does not retrain.
RUN_TRAINING = False


## Fixed compute and controls
Two cases across two folds: target-self-pretraining and pooled external pretraining, followed by identical cell fine-tuning. Each source pretraining run uses 20 epochs × 65,536 sampled windows / batch 256 = 5,120 updates. Use the last pretraining checkpoint, never choose it from scoring. Fine-tuning: maximum 100 epochs, patience 12, seed 17. External pretraining is shared between folds; target-self pretraining respects each fold's fit cutoff.


In [ ]:
from pathlib import Path
import json
import yaml
import pandas as pd
from IPython.display import display
PROJECT = Path.cwd()
if (PROJECT / 'configs/cell_multisource.yaml').exists():
    protocol = yaml.safe_load((PROJECT / 'configs/cell_multisource.yaml').read_text(encoding='utf-8'))
    display(protocol)


## Causal preprocessing
Fit one global median/mean/std per KPI on each fit prefix. Preserve observed zeros; add missing masks. A training sample contains only t−167 through t for all beams of one cell. Future labels are used only for supervised loss. Cell membership is topology, not a fitted numerical beam summary. Phase4 supplies known daily/weekly calendar positions.


## Additional data and provenance
Nine external domains: NR1800/2100/3500, CESNET-TimeSeries24, four Vigo KPIs, and CESNET-AGG23. Radio, CESNET24, Vigo and AGG23 each receive equal aggregate sampling weight, followed by entity and continuous-window sampling. They are not nine independent networks. Vigo and AGG23 add 40,653 eligible prefix entity-hours. All payload and processed checksums are pinned in preparation reports. Telecom Italia is being acquired separately and is not part of this frozen experiment.

Vigo values were scaled by the original publisher per series; its scaling fit scope is unknown. Our source task normalizes using past context only, making the numerical task invariant to positive affine scaling; this is not an audit of publisher quantization or preprocessing. Network traffic and sessions are not PRB labels. The dimensionless source head is discarded. Only temporal encoder parameters transfer, with checkpoint hashes recorded at initialization. Never use source anomaly labels to tune the target forecaster.


## Model and objective
Shared 12-hour patches with stride 6, width 64, two temporal transformer layers, and four-head cell attention. Both pretraining cases fine-tune the same full-cell architecture. The separate local/cell study supplies the spatial ablation. A causal seasonal baseline plus residual head produces nonnegative DLPRB predictions. Optimize masked MAE directly in original DLPRB units. Implementation: `src/models/cell_attention.py`; inputs: `src/features/cell_inputs.py`.


## Training and debugging
Hosted Colab/Kaggle only. Save best/last weights, optimizer and RNG each epoch to durable storage. Logs include fold/mode, parameter count, cell-window population, losses and epoch duration. Pretraining losses are dimensionless source MAE; fine-tuning/stopping losses use physical DLPRB units. Interpret historical `scaled` field names by stage. An epoch-capped run is not evidence of completed early stopping.


In [ ]:
if RUN_TRAINING:
    import subprocess, sys
    (PROJECT / 'reports').mkdir(exist_ok=True)
    with (PROJECT / 'reports/cell_multisource_training.log').open('w') as log:
        process = subprocess.Popen([sys.executable, '-u', '-m', 'scripts.train_cell_multisource'],
            stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
        for line in process.stdout:
            print(line, end='', flush=True)
            log.write(line)
            log.flush()
        TRAIN_EXIT = process.wait()
else:
    print('Review mode: training skipped.')


## Evaluation
Full frozen scoring origins and all original beams. MAE: original DLPRB units. WAPE: sum absolute error / sum absolute actual. Report all horizons, zero false positives and observed-target coverage. Independent forecast reconstruction and CPU/GPU parity are required before acceptance. Compare with the fixed current MLP-B/hurdle blend and matched local mode.


In [ ]:
# Review completed results only; no fitting or coefficient search.
for case in ('target_self_pretrain', 'multi_source_pretrain'):
    path = PROJECT / f'reports/cell_multisource_{case}_development.csv'
    if path.exists():
        results = pd.read_csv(path)
        results['WAPE (%)'] = 100 * results['wape']
        print(case)
        display(results)


## Decision and limitations
Seek at least 5% mean MAE reduction with no horizon regression; the initial 2% research gate only qualifies a candidate for further investigation. Require additional seeds and paired uncertainty checks before superiority claims. Reused development data is not fresh confirmation. Preserve negative results; no automatic promotion, final validation refit or holdout evaluation. More training does not guarantee improvement.
